# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
meta = ds.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets with their @id, name, and their fields
recordsets = meta.record_sets
print(f"Number of record sets: {len(recordsets)}\n")

for rs in recordsets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            field_obj = meta.field_by_id(field['@id']) if isinstance(field, dict) and '@id' in field else field
            if hasattr(field_obj, 'name'):
                print(f"    - @id: {getattr(field_obj, '@id', field_obj)}\tname: {field_obj.name if hasattr(field_obj, 'name') else 'N/A'}")
            else:
                print(f"    - {field_obj}")
    print()

Let's preview the records for one of the record sets. Below we iterate over the first 3 records using its `@id`.

In [ ]:
# Update to a valid record set @id from the above overview
recordset_ids = [rs['@id'] for rs in meta.record_sets]
print(f"Available RecordSet @id's:{recordset_ids}\n")
selected_recordset_id = recordset_ids[0] if recordset_ids else None

if selected_recordset_id:
    for idx, record in enumerate(ds.records(record_set=selected_recordset_id)):
        print(f"Record {idx}: {record}")
        if idx >= 2:
            break
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set and store in DataFrames
dataframes = {}

for rsid in recordset_ids:
    print(f'Loading records for RecordSet @id: {rsid}')
    records = list(ds.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample data:\n{df.head(2)}\n")

# Work with the first DataFrame
main_recordset_id = recordset_ids[0] if recordset_ids else None
if main_recordset_id:
    print(f"Columns for RecordSet '{main_recordset_id}':")
    print(dataframes[main_recordset_id].columns.tolist())
    display_df = dataframes[main_recordset_id]
    display(display_df.head())
else:
    print("No available record sets to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section includes filtering, normalization, and grouping operations by fields defined using their `@id` or column name.

In [ ]:
# Select a numeric field. Adjust to match an actual numeric column name from the record set above.
import numpy as np

df = dataframes[main_recordset_id] if main_recordset_id else pd.DataFrame()

if not df.empty:
    print("Columns and their datatypes:\n", df.dtypes)
    # Try to pick a likely numeric field
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to convert float-like columns
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='ignore')
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields detected: {numeric_candidates}")
    numeric_field = numeric_candidates[0] if numeric_candidates else None

    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        print(f"Using field '{numeric_field}' with threshold {threshold:.2f}")
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f}: {filtered_df.shape[0]} records")
        display(filtered_df.head())

        # Normalize the field
        mean_val, std_val = filtered_df[numeric_field].mean(), filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean_val) / std_val if std_val > 0 else 0.0
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical field
        possible_group_fields = df.select_dtypes(include=['object', 'category']).nunique().sort_values(ascending=True).index.tolist()
        group_field = None
        for field in possible_group_fields:
            # Skip grouping by uninformative fields
            if df[field].nunique() > 1 and df[field].nunique() < len(df) * 0.5:
                group_field = field
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(grouped_df.head())
        else:
            print("No suitable group-by field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("DataFrame for EDA is empty.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, we loaded the Croissant FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles from human mast cells using `mlcroissant`. We explored record sets and their fields, extracted data using `@id` references, filtered and normalized a numeric field, and visualized its distribution.

- For deeper insights, see the dataset documentation and field descriptions accessible via the Croissant schema or refer to the [FAIR^2 entry](https://sen.science/doi/10.71728/senscience.vq4a-28xa).
- You can extend this EDA by choosing other record sets, fields, or visualizations as needed.